# 01 · Explore the dataset

Load the **real** xLAM-60k function-calling dataset, normalize it, create train/val/test splits, and visualize its structure. No synthetic data.

In [ ]:
# --- Bootstrap: works on Kaggle, Colab, or locally ---------------------------
import os, sys, subprocess

# If you pushed this project to GitHub, set REPO_URL so Kaggle can clone it.
REPO_URL = "https://github.com/Niikhi/peft-function-calling.git"  # <-- EDIT ME
REPO_DIR = "peft-function-calling"

if not os.path.exists("src") and not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", REPO_URL], check=True)
if os.path.exists(os.path.join(REPO_DIR, "src")):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
!pip -q install pyyaml datasets matplotlib seaborn

### (Optional) log in for the gated dataset
Accept the terms on the [dataset page](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) first. Skip this and the code falls back to an ungated mirror.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from src.config import load_config
from src.data import load_records, split_records, save_jsonl

cfg = load_config()
records = load_records(cfg)          # gated -> ungated-mirror fallback
splits = split_records(cfg, records)
for name, recs in splits.items():
    save_jsonl(recs, cfg.file_in("data_dir", f"{name}.jsonl"))
print({k: len(v) for k, v in splits.items()})

### Peek at one formatted training example

In [ ]:
from transformers import AutoTokenizer
from src.data import to_openai_tool
from src.prompts import build_full_text

tok = AutoTokenizer.from_pretrained(cfg.model["hf_id"], trust_remote_code=True)
rec = splits["train"][0]
print(build_full_text(tok, rec["query"], [to_openai_tool(t) for t in rec["tools"]], rec["answers"]))

### Dataset visualizations

In [ ]:
from src.visualize import plot_dataset_eda
from IPython.display import Image
path = plot_dataset_eda(records, cfg.file_in("figures_dir", "dataset_eda.png"))
Image(path)